# 06 Matching

This notebook runs the five scripts of `06_matching/` in order with `%run`, so it runs exactly the same code as the command line (`python 06_matching/NN_name.py`). Each script's docstring says what it shows, what it needs and what it checks.

SIMBA's matching is a module of its own, `simba.Modules.Matching`. A `.def` names what to vary and what to hit, in a line's `match:` (transverse optics) or `longitudinal_match:` (the RF) block, and SIMBA sets the elements before the line is tracked. A `match:` runs in a code with a matcher: by default the line's own code, or the one named by `code:`; Ocelot, Xsuite, MAD-X, elegant and Bmad use their own matchers, Cheetah SIMBA's least-squares solver. Targets are SIMBA's Twiss names (`beta_x`, `eta_x`, `mux` in units of 2π, ...) and the transfer matrix `R11` to `R66`, in Xsuite's coordinates whatever the code. Variables are quadrupoles, any number on an element (`B1.angle`), or a group's own parameter (a chicane's `angle`); a functional definition is matched, not overwritten.

All five use JFEL from laura-lattices. `jfel.py` holds what they share: the undulator's periodic Twiss (from example 23) and a `.def` from the injector beam through the S-band linac, the X-band linearizer and the VBC chicane.

## 33 The same match in every code

Example 23's match (the S13 quads, from half strength, onto the undulator's periodic Twiss), once per code with `code:`. Every code puts the tracked beam on the target (β within 0.3%). Four quads and four targets have more than one exact solution: Ocelot, MAD-X, Cheetah, elegant and Bmad land on the strengths stored in the lattice (to $6\times10^{-6}$), Xsuite on another that meets the targets as well. A Cheetah line asked for a dispersion target, which Cheetah's optics cannot give, hands over to the first code that can, with a warning saying why.

In [ ]:
%run 33_match_in_any_code.py

## 34 Transfer-matrix targets: point-to-point imaging

`R12 = R34 = 0` from S13 to the undulator entrance. Xsuite, elegant and Bmad match on the matrix themselves; Ocelot's matcher has no matrix targets and MAD-X's refuses more variables than constraints, so SIMBA's solver works on their matrices; Cheetah declines. Each code's solution is checked by tracking: each particle's position at the entrance fitted against its coordinates at S13. The fit must carry the chromatic terms, or it returns the matrix averaged over the energy spread ($R_{12} + U_{1266}\langle\delta^2\rangle$, 0.02 to 0.26 m off here); with them the tracked $R_{12}$ and $R_{34}$ are below $4\times10^{-5}$ m, against 4.1 and 2.8 m unmatched.

In [ ]:
%run 34_matrix_targets.py

## 35 Match a chicane's R56 with its angle

`bunch_compressor.angle` as the variable: the chicane group lays its dipoles out again at every angle tried. From the design 0.13 rad onto the R56 the chicane has at 0.12 rad (example 16's exact formula), in every code: the angle comes back as 0.120000 rad, the tracked R56 is the target to $10^{-5}$, the screen in the middle straight moves where the layout puts it, the dipoles' `angle` edges follow, and the settings' `dipoleangle` keeps the new angle. The Cheetah line hands over.

In [ ]:
%run 35_chicane_r56.py

## 36 Set the linac RF for a chirp, linearised by the X-band

`longitudinal_match` (Zagorodnov and Dohlus): L02 and L03, linked by a functional definition `linac_phase`, as the main system and L4H as its fourth harmonic, for a chirp of $-15$ m$^{-1}$ with zero curvature and skewness at the nominal energy. One analytic pass is close (chirp 3% off, skewness cut fifteenfold), not exact: JFEL enters L02 at 35 MeV, where the bunch slips 1.5° against the wave, and the structure wakes take energy from the tail. Closing the loop, as the method is used in practice (track, move the targets by the miss, match again), puts the tracked beam on the targets in three passes without wakes and four with them. Tracked in Xsuite, which applies the RF exactly; `linac_phase` is what changes, so L02 and L03 stay linked.

In [ ]:
%run 36_longitudinal_match.py

## 37 Set up JFEL's bunch compression by matching

Both matches together, for a compression factor $C = 4$ with a chirp $h = -15$ m$^{-1}$: the chicane's angle for $R_{56} = (1/C - 1)/h$, its $T_{566}$ from tracking, and the RF for the chirp and the curvature $h_2 = -T_{566} h^2/R_{56}$ that cancels it, so the compression is linear: "linearise the compression, not the energy". Each particle's position after the chicane is fitted against its position before: $1/a_1 = 4.001$, and what is left curved, $a_2$, is 0.1 m$^{-1}$, against the predicted $-17.0$ m$^{-1}$ with the energy only linearised.

In [ ]:
%run 37_compression_by_matching.py